# Fortran Validation & Diagnostics

Detailed comparison between **generated** and **reference Fortran files**.

## Purpose

Identifies and analyzes differences between ACG-generated Fortran code and reference files:
- ✅ Validation of code generation
- ✅ Detailed diff analysis
- ✅ Automatic Fortran normalization
- ✅ Debugging of faulty output

**Note:** This notebook is **separate** from `multiple_species_example.ipynb` to keep the main notebooks clean.


## 1. Setup & Import Validated Helpers


In [ ]:
import sys
import warnings
from pathlib import Path

# Directories
#MODEL = 'equilibrium'
#MODEL = 'single_species'
MODEL = 'multiple_species'
#MODEL = "switches"

generated_dir = Path('generated_fortran', MODEL)
reference_dir = Path('reference_fortran', MODEL)

if not generated_dir.exists():
    warnings.warn(f"Generated directory not found: {generated_dir}")
if not reference_dir.exists():
    warnings.warn(f"Reference directory not found: {reference_dir}")

# Import validated helpers from the fortran_validation module
sys.path.insert(0, str(Path.cwd() / 'notebooks' / 'utils'))
try:
    from fortran_validation import (
        compare_fortran_files,
        validate_generated_fortran,
        print_validation_report,
    )
except ImportError as exc:
    warnings.warn(f"Could not import fortran_validation helpers: {exc}")
    compare_fortran_files = validate_generated_fortran = print_validation_report = None

print(f"Generated dir: {generated_dir}")
print(f"Reference dir: {reference_dir}")
print("Validation helpers loaded" if compare_fortran_files else "⚠ Validation helpers NOT loaded")
print()

# List of files to validate
validation_files = [
    'common_geo.inc',
    'boundaries.f',
    'molecular.f',
    'biogeo.f',
    'basic.f',
    'issolid.f',
    'notransport.f',
    'rates.f',
    'jacobian.f',
    'residual.f',
    'ssrates.f',
    'initialcond.f',
    'output.f',
    'switches.f'
]

print(f"Files to validate: {len(validation_files)}")
for i, fname in enumerate(validation_files, 1):
    print(f"  {i:2d}. {fname}")


## 2. Automatic Normalization

The `fortran_validation` helpers automatically perform the following normalizations:

### F77 fixed-format normalization
- Remove line numbers/labels (columns 1-5)
- Normalize indentation
- Join continuation lines (`&`, `+`)

### Whitespace & syntax normalization
- Spaces around operators (`=`, `+`, `-`, `*`, `/`)
- Spaces after commas
- Fortran keywords: `endif` → `end if`
- Fortran operators: `.AND.`, `.OR.` → lowercase
- Comment formatting

**Real code differences are NOT ignored!**


## 3. Global Validation of All Files


In [ ]:
# Run validation
if validate_generated_fortran is not None:
    print("Validating all generated Fortran files...")
    print()

    results = validate_generated_fortran(
        generated_dir=generated_dir,
        reference_dir=reference_dir,
        file_list=validation_files,
        verbose=False
    )

    # Print report
    all_passed = print_validation_report(results, "Fortran Validation")

    if all_passed:
        print("\n🎉 ALL VALIDATIONS PASSED!")
    else:
        print("\n⚠️  Some files differ. See details above.")
else:
    warnings.warn("Skipping validation: fortran_validation helpers were not loaded.")


## 4. Detailed Analysis: Compare a Single File


In [ ]:
# Choose a file for detailed comparison
file_to_compare = 'biogeo.f'  # change as needed: 'residual.f', 'rates.f', etc.

gen_file = generated_dir / file_to_compare
ref_file = reference_dir / file_to_compare

if gen_file.exists() and ref_file.exists() and compare_fortran_files is not None:
    print(f"\nDETAILED ANALYSIS: {file_to_compare}")
    print("="*80)

    is_identical, error_msg = compare_fortran_files(gen_file, ref_file, verbose=True)

    if is_identical:
        print(f"\n✅ {file_to_compare} is identical!")
    else:
        print(f"\n❌ {file_to_compare} differs:")
        print(error_msg)
else:
    if not gen_file.exists():
        warnings.warn(f"Generated file not found: {gen_file}")
    if not ref_file.exists():
        warnings.warn(f"Reference file not found: {ref_file}")
    if compare_fortran_files is None:
        warnings.warn("Skipping comparison: fortran_validation helpers were not loaded.")


In [ ]:
# Symbolic equivalence check per isp for ssrates.f (rat/drdc)
import re
import warnings
import sympy as sp
from pathlib import Path

file_to_compare = 'ssrates.f'  
gen_path = generated_dir / file_to_compare
ref_path = reference_dir / file_to_compare

def fortran_statements(path: Path):
    lines = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    stmts, cur = [], ''
    for raw in lines:
        s = raw.rstrip()
        if not s.strip():
            continue
        if s.lstrip().startswith(('c', 'C', '!', '*')):
            continue
        is_cont = (len(s) >= 6 and s[:5] == '     ' and s[5] in ('+', '&'))
        payload = s[6:] if len(s) > 6 else ''
        if is_cont:
            cur += payload.strip()
        else:
            if cur:
                stmts.append(cur.strip())
            cur = (s[6:] if len(s) > 6 else s).strip()
    if cur:
        stmts.append(cur.strip())
    return stmts

def extract_isp_exprs(path: Path):
    stmts = [x.lower() for x in fortran_statements(path)]
    out, current = {}, None
    for st in stmts:
        m = re.match(r'if\s*\(\s*isp\.eq\.(\d+)\s*\)\s*then', st)
        if m:
            current = int(m.group(1))
            out[current] = {'rat': None, 'drdc': None}
            continue
        if st.startswith('end if') or st == 'endif':
            current = None
            continue
        if current is not None:
            if st.startswith('rat=') or st.startswith('rat ='):
                out[current]['rat'] = st.split('=', 1)[1].strip()
            elif st.startswith('drdc=') or st.startswith('drdc ='):
                out[current]['drdc'] = st.split('=', 1)[1].strip()
    return out

def to_sympy(expr: str):
    if expr is None:
        return None
    x = expr
    x = re.sub(r'((?:\d+\.\d*)|(?:\d+)|(?:\.\d+))[dD]([+-]?\d+)', r'\1e\2', x)
    x = re.sub(r'\bdexp\s*\(', 'exp(', x)
    x = re.sub(r'\bdlog\s*\(', 'log(', x)
    x = re.sub(r'\bdsqrt\s*\(', 'sqrt(', x)
    x = re.sub(r'\bdabs\s*\(', 'Abs(', x)
    x = re.sub(r'\bdmin1\s*\(', 'Min(', x)
    x = re.sub(r'\bdmax1\s*\(', 'Max(', x)
    x = re.sub(r'sp\s*\(\s*(\d+)\s*,\s*j\s*\)', r'sp_\1', x, flags=re.IGNORECASE)
    x = re.sub(r'spold\s*\(\s*(\d+)\s*,\s*j\s*\)', r'spold_\1', x, flags=re.IGNORECASE)
    x = x.replace('^', '**')
    function_names = set(re.findall(r'\b([A-Za-z_][A-Za-z0-9_]*)\s*\(', x))
    names = set(re.findall(r'[A-Za-z_][A-Za-z0-9_]*', x)) - function_names
    names.discard('e')
    sym = {n: sp.Symbol(n) for n in names}
    sym.update({
        'exp': sp.exp,
        'log': sp.log,
        'sqrt': sp.sqrt,
        'abs': sp.Abs,
        'Abs': sp.Abs,
        'Min': sp.Min,
        'Max': sp.Max,
        'Rational': sp.Rational,
    })
    for name in function_names:
        if name not in sym:
            sym[name] = sp.Function(name)
    try:
        return sp.sympify(x, locals=sym)
    except Exception:
        return None  # Parse error -> reported as non-equivalent

if not gen_path.exists() or not ref_path.exists():
    if not gen_path.exists():
        warnings.warn(f"Generated file not found: {gen_path}")
    if not ref_path.exists():
        warnings.warn(f"Reference file not found: {ref_path}")
else:
    gen = extract_isp_exprs(gen_path)
    ref = extract_isp_exprs(ref_path)
    isp_values = sorted(set(gen) | set(ref))

    print('Symbolic equivalence per isp (sympy.simplify(lhs-rhs))')
    print('-' * 80)
    not_equiv = []
    for isp in isp_values:
        gr = to_sympy(gen.get(isp, {}).get('rat'))
        rr = to_sympy(ref.get(isp, {}).get('rat'))
        gd = to_sympy(gen.get(isp, {}).get('drdc'))
        rd = to_sympy(ref.get(isp, {}).get('drdc'))

        rat_ok = False
        drdc_ok = False

        if gr is not None and rr is not None:
            rat_diff = sp.simplify(sp.expand(gr - rr))
            rat_ok = (rat_diff == 0)
        else:
            rat_diff = None

        if gd is not None and rd is not None:
            drdc_diff = sp.simplify(sp.expand(gd - rd))
            drdc_ok = (drdc_diff == 0)
        else:
            drdc_diff = None

        status = 'OK' if (rat_ok and drdc_ok) else 'DIFF'
        print(f'isp={isp:2d}: {status} (rat={rat_ok}, drdc={drdc_ok})')
        if not (rat_ok and drdc_ok):
            not_equiv.append((isp, rat_diff, drdc_diff))

    print('-' * 80)
    print(f'Equivalent blocks: {len(isp_values) - len(not_equiv)}/{len(isp_values)}')
    print('Non-equivalent isp:', [i for i, _, _ in not_equiv])

    for isp, rat_diff, drdc_diff in not_equiv[:6]:
        print(f'\nisp={isp} differences:')
        if rat_diff is not None:
            print('  rat diff:', str(rat_diff)[:500])
        if drdc_diff is not None:
            print('  drdc diff:', str(drdc_diff)[:500])


In [ ]:
# Symbolic comparison for residual.f (funcs(i)) – validation only, no regeneration
import re
import warnings
import sympy as sp
from pathlib import Path

file_to_compare = 'residual.f'  
gen_path = generated_dir / file_to_compare
ref_path = reference_dir / file_to_compare
    
def fortran_statements(path: Path):
    lines = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    stmts, cur = [], ''
    for raw in lines:
        s = raw.rstrip()
        if not s.strip():
            continue
        if s.lstrip().startswith(('c', 'C', '!', '*')):
            continue
        is_cont = (len(s) >= 6 and s[:5] == '     ' and s[5] in ('+', '&'))
        payload = s[6:] if len(s) > 6 else ''
        if is_cont:
            cur += payload.strip()
        else:
            if cur:
                stmts.append(cur.strip())
            cur = (s[6:] if len(s) > 6 else s).strip()
    if cur:
        stmts.append(cur.strip())
    return stmts

def extract_funcs(path: Path):
    out = {}
    for st in [x.lower() for x in fortran_statements(path)]:
        m = re.match(r'funcs\((\d+)\)\s*=\s*(.*)$', st)
        if m:
            out[int(m.group(1))] = m.group(2).strip()
    return out

def to_sympy(expr: str):
    if expr is None:
        return None
    x = expr
    x = re.sub(r'((?:\d+\.\d*)|(?:\d+)|(?:\.\d+))[dD]([+-]?\d+)', r'\1e\2', x)
    x = re.sub(r'\bdexp\s*\(', 'exp(', x)
    x = re.sub(r'\bdlog\s*\(', 'log(', x)
    x = re.sub(r'\bdsqrt\s*\(', 'sqrt(', x)
    x = re.sub(r'\bdabs\s*\(', 'Abs(', x)
    x = re.sub(r'\bdmin1\s*\(', 'Min(', x)
    x = re.sub(r'\bdmax1\s*\(', 'Max(', x)
    x = re.sub(r'sp\s*\(\s*(\d+)\s*,\s*j\s*\)', r'sp_\1', x, flags=re.IGNORECASE)
    x = re.sub(r'spold\s*\(\s*(\d+)\s*,\s*j\s*\)', r'spold_\1', x, flags=re.IGNORECASE)
    x = x.replace('^', '**')
    function_names = set(re.findall(r'\b([A-Za-z_][A-Za-z0-9_]*)\s*\(', x))
    names = set(re.findall(r'[A-Za-z_][A-Za-z0-9_]*', x)) - function_names
    names.discard('e')
    sym = {n: sp.Symbol(n) for n in names}
    sym.update({
        'exp': sp.exp,
        'log': sp.log,
        'sqrt': sp.sqrt,
        'abs': sp.Abs,
        'Abs': sp.Abs,
        'Min': sp.Min,
        'Max': sp.Max,
        'Rational': sp.Rational,
    })
    for name in function_names:
        if name not in sym:
            sym[name] = sp.Function(name)
    try:
        return sp.sympify(x, locals=sym)
    except Exception:
        return None

if not gen_path.exists() or not ref_path.exists():
    if not gen_path.exists():
        warnings.warn(f"Generated file not found: {gen_path}")
    if not ref_path.exists():
        warnings.warn(f"Reference file not found: {ref_path}")
else:
    g = extract_funcs(gen_path)
    r = extract_funcs(ref_path)
    func_indices = sorted(set(g) | set(r))

    not_equiv = []
    details = {}
    for i in func_indices:
        if i not in g or i not in r:
            not_equiv.append(i)
            details[i] = 'MISSING'
            continue
        diff = sp.simplify(sp.expand(to_sympy(g[i]) - to_sympy(r[i])))
        if diff != 0:
            not_equiv.append(i)
            details[i] = diff

    print('Residual symbolic eq (funcs):', f"{len(func_indices)-len(not_equiv)}/{len(func_indices)}")
    print('Non-equivalent funcs:', not_equiv)

    # Show generated and reference expressions for the first 3 non-equivalent funcs
    for i in not_equiv[:3]:
        print(f'\n--- funcs({i}) ---')
        print(f'  GEN: {g.get(i, "<missing>")[:200]}')
        print(f'  REF: {r.get(i, "<missing>")[:200]}')
        if isinstance(details.get(i), sp.Basic):
            print(f'  DIFF (simplified): {str(details[i])[:300]}')


In [ ]:
# Symbolic comparison for jacobian.f – pd(i,j) dict-based, order-independent
# Both files may list entries in a different order → compare via a {(i,j): expr} dict.
# Exact rational arithmetic (fractions.Fraction) avoids floating-point noise (as seen with funcs(2)).
import re
import warnings
import sympy as sp
import fractions
from pathlib import Path

file_to_compare = 'jacobian.f'  
gen_path = generated_dir / file_to_compare
ref_path = reference_dir / file_to_compare

def fortran_statements(path: Path):
    """Joins Fortran fixed-form continuation lines into complete statements."""
    lines = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    stmts, cur = [], ''
    for raw in lines:
        s = raw.rstrip()
        if not s.strip():
            continue
        if s.lstrip().startswith(('c', 'C', '!', '*')):
            continue
        is_cont = (len(s) >= 6 and s[:5] == '     ' and s[5] in ('+', '&'))
        payload = s[6:] if len(s) > 6 else ''
        if is_cont:
            cur += payload.strip()
        else:
            if cur:
                stmts.append(cur.strip())
            cur = (s[6:].strip() if len(s) > 6 else s.strip())
    if cur:
        stmts.append(cur.strip())
    return stmts

def extract_pd(path: Path):
    """Extracts all pd(i,j) = expr assignments as a dict {(i,j): str}."""
    out = {}
    for st in fortran_statements(path):
        m = re.match(r'pd\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)\s*=\s*(.*)$', st, re.IGNORECASE)
        if m:
            key = (int(m.group(1)), int(m.group(2)))
            out[key] = m.group(3).strip()
    return out

def to_sympy_exact(expr: str):
    """Converts a Fortran expression to SymPy using exact rational literals (no float error)."""
    if expr is None:
        return sp.Integer(0)
    x = expr.lower()

    # Fortran D-notation → exact Python fractions, e.g. 21.d0 -> 21, 0.1d1 -> 1, 21.d0/968.d0 -> Rational(21,968)
    def fort_to_rational(m):
        mantissa_str, exp_str = m.group(1), m.group(2)
        try:
            f = fractions.Fraction(mantissa_str) * fractions.Fraction(10) ** int(exp_str)
            return f'Rational({f.numerator},{f.denominator})' if f.denominator != 1 else str(f.numerator)
        except Exception:
            return repr(float(mantissa_str) * 10 ** int(exp_str))

    # Decimal d-notation: 21.d0, 0.1d1, 1.5d-3 (requires a decimal point)
    x = re.sub(r'([\d]+\.[\d]*|[\d]*\.[\d]+)[dD]([+\-]?\d+)', fort_to_rational, x)
    # Integer d-notation: 1d0, 2d0 -> 1, 2 (no decimal point; only if not already substituted)
    x = re.sub(r'(?<![a-z_])(\d+)[dD]([+\-]?\d+)(?![a-z_0-9])', fort_to_rational, x)
    x = re.sub(r'\bdexp\s*\(', 'exp(', x)
    x = re.sub(r'\bdlog\s*\(', 'log(', x)
    x = re.sub(r'\bdsqrt\s*\(', 'sqrt(', x)
    x = re.sub(r'\bdabs\s*\(', 'Abs(', x)
    x = re.sub(r'\bdmin1\s*\(', 'Min(', x)
    x = re.sub(r'\bdmax1\s*\(', 'Max(', x)
    x = re.sub(r'sp\s*\(\s*(\d+)\s*,\s*j\s*\)',    r'sp_\1',    x)
    x = re.sub(r'spold\s*\(\s*(\d+)\s*,\s*j\s*\)', r'spold_\1', x)
    x = x.replace('^', '**')

    function_names = set(re.findall(r'\b([A-Za-z_][A-Za-z0-9_]*)\s*\(', x))
    names = set(re.findall(r'[A-Za-z_][A-Za-z0-9_]*', x)) - function_names
    names.discard('e')
    sym = {n: sp.Symbol(n) for n in names}
    sym.update({
        'exp': sp.exp,
        'log': sp.log,
        'sqrt': sp.sqrt,
        'abs': sp.Abs,
        'Abs': sp.Abs,
        'Min': sp.Min,
        'Max': sp.Max,
        'Rational': sp.Rational,
    })
    for name in function_names:
        if name not in sym:
            sym[name] = sp.Function(name)
    try:
        return sp.sympify(x, locals=sym)
    except Exception:
        return None  # Parse error -> reported as non-equivalent

if not gen_path.exists() or not ref_path.exists():
    if not gen_path.exists():
        warnings.warn(f"Generated file not found: {gen_path}")
    if not ref_path.exists():
        warnings.warn(f"Reference file not found: {ref_path}")
else:
    # --- Parse files ---
    gen_pd = extract_pd(gen_path)
    ref_pd = extract_pd(ref_path)

    print(f'Entries generated : {len(gen_pd)}')
    print(f'Entries reference : {len(ref_pd)}')
    all_keys = sorted(set(gen_pd) | set(ref_pd))
    print(f'Unique (i,j)      : {len(all_keys)}')
    print()

    not_equiv  = []   # [(key, diff_expr, gen_str, ref_str)]
    missing_gen = []  # in REF but not in GEN
    missing_ref = []  # in GEN but not in REF

    for key in all_keys:
        g_str = gen_pd.get(key)
        r_str = ref_pd.get(key)
        if g_str is None:
            missing_gen.append(key)
            continue
        if r_str is None:
            missing_ref.append(key)
            continue
        ge = to_sympy_exact(g_str)
        re_ = to_sympy_exact(r_str)
        if ge is None or re_ is None:
            not_equiv.append((key, None, g_str, r_str))
            continue
        diff = sp.simplify(sp.expand(ge - re_))
        if diff != 0:
            not_equiv.append((key, diff, g_str, r_str))

    n_ok = len(all_keys) - len(not_equiv) - len(missing_gen) - len(missing_ref)
    print(f'Jacobian symbolically equivalent: {n_ok}/{len(all_keys)} entries OK')
    if not_equiv:
        print(f'Not equivalent   : {[k for k,*_ in not_equiv]}')
    if missing_gen:
        print(f'Missing in GEN   : {missing_gen}')
    if missing_ref:
        print(f'Missing in REF   : {missing_ref}')

    # Details for the first 4 differing entries
    for key, diff, g_str, r_str in not_equiv[:4]:
        print(f'\n--- pd{key} ---')
        print(f'  GEN : {g_str[:250]}')
        print(f'  REF : {r_str[:250]}')
        if diff is not None:
            print(f'  DIFF: {str(diff)[:300]}')


In [ ]:
# Symbolic comparison for rates.f (r(i,j))
import re
import warnings
import sympy as sp
from pathlib import Path

file_to_compare = 'rates.f'  
gen_path = generated_dir / file_to_compare
ref_path = reference_dir / file_to_compare

def fortran_statements(path: Path):
    lines = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    stmts, cur = [], ''
    for raw in lines:
        s = raw.rstrip()
        if not s.strip():
            continue
        if s.lstrip().startswith(('c', 'C', '!', '*')):
            continue
        is_cont = (len(s) >= 6 and s[:5] == '     ' and s[5] in ('+', '&'))
        payload = s[6:] if len(s) > 6 else ''
        if is_cont:
            cur += payload.strip()
        else:
            if cur:
                stmts.append(cur.strip())
            cur = (s[6:] if len(s) > 6 else s).strip()
    if cur:
        stmts.append(cur.strip())
    return stmts

def extract_rates(path: Path):
    out = {}
    for st in [x.lower() for x in fortran_statements(path)]:
        m = re.match(r'r\((\d+)\s*,\s*j\)\s*=\s*(.*)$', st)
        if m:
            out[int(m.group(1))] = m.group(2).strip()
    return out

def to_sympy(expr: str):
    if expr is None:
        return None
    x = expr
    x = re.sub(r'((?:\d+\.\d*)|(?:\d+)|(?:\.\d+))[dD]([+-]?\d+)', r'\1e\2', x)
    x = re.sub(r'\bdexp\s*\(', 'exp(', x)
    x = re.sub(r'\bdlog\s*\(', 'log(', x)
    x = re.sub(r'\bdsqrt\s*\(', 'sqrt(', x)
    x = re.sub(r'\bdabs\s*\(', 'Abs(', x)
    x = re.sub(r'\bdmin1\s*\(', 'Min(', x)
    x = re.sub(r'\bdmax1\s*\(', 'Max(', x)
    x = re.sub(r'sp\s*\(\s*(\d+)\s*,\s*j\s*\)', r'sp_\1', x, flags=re.IGNORECASE)
    x = re.sub(r'spold\s*\(\s*(\d+)\s*,\s*j\s*\)', r'spold_\1', x, flags=re.IGNORECASE)
    x = x.replace('^', '**')
    function_names = set(re.findall(r'\b([A-Za-z_][A-Za-z0-9_]*)\s*\(', x))
    names = set(re.findall(r'[A-Za-z_][A-Za-z0-9_]*', x)) - function_names
    names.discard('e')
    sym = {n: sp.Symbol(n) for n in names}
    sym.update({
        'exp': sp.exp,
        'log': sp.log,
        'sqrt': sp.sqrt,
        'abs': sp.Abs,
        'Abs': sp.Abs,
        'Min': sp.Min,
        'Max': sp.Max,
        'Rational': sp.Rational,
    })
    for name in function_names:
        if name not in sym:
            sym[name] = sp.Function(name)
    try:
        return sp.sympify(x, locals=sym)
    except Exception:
        return None

if not gen_path.exists() or not ref_path.exists():
    if not gen_path.exists():
        warnings.warn(f"Generated file not found: {gen_path}")
    if not ref_path.exists():
        warnings.warn(f"Reference file not found: {ref_path}")
else:
    g = extract_rates(gen_path)
    r = extract_rates(ref_path)
    rate_indices = sorted(set(g) | set(r))

    not_equiv = []
    details = {}
    for i in rate_indices:
        if i not in g or i not in r:
            not_equiv.append(i)
            details[i] = 'MISSING'
            continue
        diff = sp.simplify(sp.expand(to_sympy(g[i]) - to_sympy(r[i])))
        if diff != 0:
            not_equiv.append(i)
            details[i] = diff

    print('Rates symbolic eq (r(i,j)):', f"{len(rate_indices)-len(not_equiv)}/{len(rate_indices)}")
    print(f'Differing rates: {len(not_equiv)}/{len(rate_indices)}')
    print('Non-equivalent rates:', not_equiv)

    for i in not_equiv[:5]:
        print(f'\n--- r({i},j) ---')
        print(f'  GEN: {g.get(i, "<missing>")[:220]}')
        print(f'  REF: {r.get(i, "<missing>")[:220]}')
        if isinstance(details.get(i), sp.Basic):
            print(f'  DIFF (simplified): {str(details[i])[:300]}')


## 5. Additional Symbolic Checks

Scalar assignments in `biogeo.f`, boundary values in `boundaries.f`, and the `spi(i)` vector in `initialcond.f` are compared symbolically in the same way as the earlier blocks.


In [ ]:
# Symbolic comparison for biogeo.f assignments
import re
import warnings
import sympy as sp
from pathlib import Path

file_to_compare = 'biogeo.f'
gen_path = generated_dir / file_to_compare
ref_path = reference_dir / file_to_compare

# Join continuation lines and keep only non-comment assignments

def fortran_statements(path: Path):
    lines = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    stmts, cur = [], ''
    for raw in lines:
        s = raw.rstrip()
        if not s.strip():
            continue
        if s.lstrip().startswith(('c', 'C', '!', '*')):
            continue
        is_cont = (len(s) >= 6 and s[:5] == '     ' and s[5] in ('+', '&'))
        payload = s[6:] if len(s) > 6 else ''
        if is_cont:
            cur += payload.strip()
        else:
            if cur:
                stmts.append(cur.strip())
            cur = (s[6:].strip() if len(s) > 6 else s.strip())
    if cur:
        stmts.append(cur.strip())
    return stmts


def extract_scalar_assignments(path: Path):
    out = {}
    for st in [x.lower() for x in fortran_statements(path)]:
        m = re.match(r'([a-z_][a-z0-9_]*)\s*=\s*(.*)$', st)
        if m:
            name, expr = m.group(1), m.group(2).strip()
            if name not in {'end', 'subroutine', 'include', 'if', 'do', 'continue'}:
                out[name] = expr
    return out


def to_sympy(expr: str):
    x = expr.strip()
    if not x:
        return None
    x = x.lower()
    x = re.sub(r'((?:\d+\.\d*)|(?:\d+)|(?:\.\d+))[dD]([+-]?\d+)', r'\1e\2', x)
    x = re.sub(r'\bdexp\s*\(', 'exp(', x)
    x = re.sub(r'\bdlog\s*\(', 'log(', x)
    x = re.sub(r'\bdsqrt\s*\(', 'sqrt(', x)
    x = re.sub(r'\bdabs\s*\(', 'Abs(', x)
    x = re.sub(r'\bdmin1\s*\(', 'Min(', x)
    x = re.sub(r'\bdmax1\s*\(', 'Max(', x)
    x = x.replace('^', '**')
    function_names = set(re.findall(r'\b([A-Za-z_][A-Za-z0-9_]*)\s*\(', x))
    names = set(re.findall(r'[A-Za-z_][A-Za-z0-9_]*', x)) - function_names
    names.discard('e')
    sym = {n: sp.Symbol(n) for n in names}
    sym.update({
        'exp': sp.exp,
        'log': sp.log,
        'sqrt': sp.sqrt,
        'abs': sp.Abs,
        'Abs': sp.Abs,
        'Min': sp.Min,
        'Max': sp.Max,
        'Rational': sp.Rational,
    })
    for name in function_names:
        if name not in sym:
            sym[name] = sp.Function(name)
    try:
        return sp.sympify(x, locals=sym)
    except Exception:
        return None

if not gen_path.exists() or not ref_path.exists():
    if not gen_path.exists():
        warnings.warn(f'Generated file not found: {gen_path}')
    if not ref_path.exists():
        warnings.warn(f'Reference file not found: {ref_path}')
else:
    gen = extract_scalar_assignments(gen_path)
    ref = extract_scalar_assignments(ref_path)
    common = sorted(set(gen) & set(ref))
    print('biogeo scalar assignments:', len(common), '/', len(common))
    non_equiv = []
    for name in common:
        diff = sp.simplify(sp.expand(to_sympy(gen[name]) - to_sympy(ref[name])))
        if diff != 0:
            non_equiv.append((name, diff))
    print(f'Differing scalar variables: {len(non_equiv)}/{len(common)}')
    print('Non-equivalent scalar variables:', [n for n, _ in non_equiv])
    for name, diff in non_equiv[:5]:
        print(f'--- {name} ---')
        print('  GEN:', gen[name][:200])
        print('  REF:', ref[name][:200])
        print('  DIFF:', str(diff)[:300])


In [ ]:
# Symbolic comparison for boundaries.f (spb(:,1), spb(:,2), ibc(:,1), ibc(:,2))
import re
import warnings
import sympy as sp
from pathlib import Path

file_to_compare = 'boundaries.f'
gen_path = generated_dir / file_to_compare
ref_path = reference_dir / file_to_compare


def fortran_statements(path: Path):
    lines = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    stmts, cur = [], ''
    for raw in lines:
        s = raw.rstrip()
        if not s.strip():
            continue
        if s.lstrip().startswith(('c', 'C', '!', '*')):
            continue
        is_cont = (len(s) >= 6 and s[:5] == '     ' and s[5] in ('+', '&'))
        payload = s[6:] if len(s) > 6 else ''
        if is_cont:
            cur += payload.strip()
        else:
            if cur:
                stmts.append(cur.strip())
            cur = (s[6:] if len(s) > 6 else s).strip()
    if cur:
        stmts.append(cur.strip())
    return stmts


def extract_boundary_values(path: Path):
    out = {}
    for st in [x.lower() for x in fortran_statements(path)]:
        m = re.match(r'spb\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)\s*=\s*(.*)$', st)
        if m:
            key = ('spb', int(m.group(1)), int(m.group(2)))
            out[key] = m.group(3).strip()
        m = re.match(r'ibc\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)\s*=\s*(.*)$', st)
        if m:
            key = ('ibc', int(m.group(1)), int(m.group(2)))
            out[key] = m.group(3).strip()
    return out


def to_sympy(expr: str):
    x = expr.strip()
    if not x:
        return None
    x = x.lower()
    x = re.sub(r'((?:\d+\.\d*)|(?:\d+)|(?:\.\d+))[dD]([+-]?\d+)', r'\1e\2', x)
    x = re.sub(r'\bdexp\s*\(', 'exp(', x)
    x = re.sub(r'\bdlog\s*\(', 'log(', x)
    x = re.sub(r'\bdsqrt\s*\(', 'sqrt(', x)
    x = re.sub(r'\bdabs\s*\(', 'Abs(', x)
    x = re.sub(r'\bdmin1\s*\(', 'Min(', x)
    x = re.sub(r'\bdmax1\s*\(', 'Max(', x)
    x = x.replace('^', '**')
    function_names = set(re.findall(r'\b([A-Za-z_][A-Za-z0-9_]*)\s*\(', x))
    names = set(re.findall(r'[A-Za-z_][A-Za-z0-9_]*', x)) - function_names
    names.discard('e')
    sym = {n: sp.Symbol(n) for n in names}
    sym.update({
        'exp': sp.exp,
        'log': sp.log,
        'sqrt': sp.sqrt,
        'abs': sp.Abs,
        'Abs': sp.Abs,
        'Min': sp.Min,
        'Max': sp.Max,
        'Rational': sp.Rational,
    })
    for name in function_names:
        if name not in sym:
            sym[name] = sp.Function(name)
    try:
        return sp.sympify(x, locals=sym)
    except Exception:
        return None

if not gen_path.exists() or not ref_path.exists():
    if not gen_path.exists():
        warnings.warn(f'Generated file not found: {gen_path}')
    if not ref_path.exists():
        warnings.warn(f'Reference file not found: {ref_path}')
else:
    gen = extract_boundary_values(gen_path)
    ref = extract_boundary_values(ref_path)
    all_keys = sorted(set(gen) | set(ref), key=lambda key: str(key))
    print('Boundary entries:', len(all_keys))
    non_equiv = []
    for key in all_keys:
        if key not in gen or key not in ref:
            non_equiv.append((key, 'MISSING'))
            continue
        ge = to_sympy(gen[key])
        re_ = to_sympy(ref[key])
        if ge is None or re_ is None:
            non_equiv.append((key, 'PARSE_ERROR'))
            continue
        diff = sp.simplify(sp.expand(ge - re_))
        if diff != 0:
            non_equiv.append((key, diff))
    print('Non-equivalent boundary entries:', [k for k, _ in non_equiv])
    for key, diff in non_equiv[:6]:
        print(f'--- {key} ---')
        print('  GEN:', str(gen.get(key, '<missing>'))[:200])
        print('  REF:', str(ref.get(key, '<missing>'))[:200])
        if isinstance(diff, sp.Basic):
            print('  DIFF:', str(diff)[:300])


In [ ]:
# Symbolic comparison for initialcond.f spi(i)
import re
import warnings
import sympy as sp
from pathlib import Path

file_to_compare = 'initialcond.f'
gen_path = generated_dir / file_to_compare
ref_path = reference_dir / file_to_compare


def fortran_statements(path: Path):
    lines = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    stmts, cur = [], ''
    for raw in lines:
        s = raw.rstrip()
        if not s.strip():
            continue
        if s.lstrip().startswith(('c', 'C', '!', '*')):
            continue
        is_cont = (len(s) >= 6 and s[:5] == '     ' and s[5] in ('+', '&'))
        payload = s[6:] if len(s) > 6 else ''
        if is_cont:
            cur += payload.strip()
        else:
            if cur:
                stmts.append(cur.strip())
            cur = (s[6:] if len(s) > 6 else s).strip()
    if cur:
        stmts.append(cur.strip())
    return stmts


def extract_spi(path: Path):
    out = {}
    for st in [x.lower() for x in fortran_statements(path)]:
        m = re.match(r'spi\s*\(\s*(\d+)\s*\)\s*=\s*(.*)$', st)
        if m:
            out[int(m.group(1))] = m.group(2).strip()
    return out


def to_sympy(expr: str):
    x = expr.strip()
    if not x:
        return None
    x = x.lower()
    x = re.sub(r'((?:\d+\.\d*)|(?:\d+)|(?:\.\d+))[dD]([+-]?\d+)', r'\1e\2', x)
    x = x.replace('^', '**')
    names = set(re.findall(r'[A-Za-z_][A-Za-z0-9_]*', x))
    names.discard('e')
    sym = {n: sp.Symbol(n) for n in names}
    return sp.sympify(x, locals=sym)

if not gen_path.exists() or not ref_path.exists():
    if not gen_path.exists():
        warnings.warn(f'Generated file not found: {gen_path}')
    if not ref_path.exists():
        warnings.warn(f'Reference file not found: {ref_path}')
else:
    gen = extract_spi(gen_path)
    ref = extract_spi(ref_path)
    all_keys = sorted(set(gen) | set(ref))
    print('spi entries:', len(all_keys))
    non_equiv = []
    for i in all_keys:
        if i not in gen or i not in ref:
            non_equiv.append((i, 'MISSING'))
            continue
        ge = to_sympy(gen[i])
        re_ = to_sympy(ref[i])
        if ge is None or re_ is None:
            non_equiv.append((i, 'PARSE_ERROR'))
            continue
        diff = sp.simplify(sp.expand(ge - re_))
        if diff != 0:
            non_equiv.append((i, diff))
    print('Non-equivalent spi entries:', [i for i, _ in non_equiv])
    for i, diff in non_equiv[:6]:
        print(f'--- spi({i}) ---')
        print('  GEN:', gen.get(i, '<missing>')[:200])
        print('  REF:', ref.get(i, '<missing>')[:200])
        if isinstance(diff, sp.Basic):
            print('  DIFF:', str(diff)[:300])
